# VoxaVersa: train all AI voices on a free Colab GPU

This notebook runs **the same training queue** as VoxaVersa's CPU trainer, on Google's GPU: same data, same
quality checks every 10 epochs, same automatic stop and best-checkpoint choice, and the same matching of every
singer to the built-in genre voices. It is just much faster (about 1 - 3 hours per voice instead of 14 - 45).

**Before you start** (once): put `VoxaVersa_Colab_Data.zip` in your Google Drive, in a folder named
`VoxaVersa` (so: *My Drive > VoxaVersa > VoxaVersa_Colab_Data.zip*). Don't unzip it. It is private
test/training material: don't share it.

**Every time**

1. Menu **Runtime > Change runtime type > T4 GPU > Save**.
2. Menu **Runtime > Run all**. Allow access to Google Drive when asked.
3. Step 6 shows the progress. Leave the tab open. If Colab disconnects or the daily GPU time runs out, come
   back later and do **Run all** again: it continues where it stopped (from the last checkpoint on Drive).

Finished voices appear in *My Drive > VoxaVersa > queue > voices* as `.vxpack` files (step 7 lists them and
plays before/after samples). Step-by-step guide: `docs/COLAB.md` in the VoxaVersa repository.

In [ ]:
#@title 1. Check the GPU
import subprocess
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
if gpu:
    print("OK - GPU found:", gpu)
else:
    raise SystemExit("No GPU. Menu Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all again.")

In [ ]:
#@title 2. Settings and Google Drive
#@markdown **Training data kit** in your Google Drive (path inside *My Drive*):
DATA_KIT = "VoxaVersa/VoxaVersa_Colab_Data.zip"  #@param {type:"string"}
#@markdown Also train the 20 free VocalSet singers (downloaded here; CC BY 4.0). Untick to train only the Cymatics singers.
INCLUDE_VOCALSET = True  #@param {type:"boolean"}
#@markdown Precision: *mixed* is RVC's standard GPU training (faster); *full* is fp32 like the CPU trainer (~2x slower).
PRECISION = "mixed"  #@param ["mixed", "full"]
BATCH_SIZE = 8  #@param {type:"integer"}
#@markdown Where the VoxaVersa tools come from (leave as is unless told otherwise):
VOXAVERSA_REPO = "https://github.com/DK7602/VoxaVersa"  #@param {type:"string"}
VOXAVERSA_BRANCH = "claude/voxaversa-voice-plugin-20arm6"  #@param {type:"string"}

import os
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/VoxaVersa/queue"
KIT = os.path.join("/content/drive/MyDrive", DATA_KIT)
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(KIT):
    raise SystemExit("Not found in your Drive: My Drive/%s\nUpload VoxaVersa_Colab_Data.zip there first." % DATA_KIT)
print("OK - data kit found; results and checkpoints go to My Drive/VoxaVersa/queue")

In [ ]:
#@title 3. Install the training software (about 5 minutes)
import os, subprocess

def sh(cmd, quiet=True):
    p = subprocess.run(cmd, shell=True, text=True, capture_output=quiet)
    if p.returncode != 0:
        if quiet:
            print(p.stdout[-3000:], p.stderr[-3000:])
        raise SystemExit("Step failed: " + cmd)

REPO = "/content/VoxaVersa"
if not os.path.isfile(REPO + "/tools/training/colab_queue.py"):
    sh("rm -rf %s && git clone -q --depth 1 --branch %s %s %s" % (REPO, VOXAVERSA_BRANCH, VOXAVERSA_REPO, REPO))
else:
    sh("git -C %s pull -q" % REPO)
TOOLS = REPO + "/tools/training"

# The same package versions as the CPU trainer (PyTorch 2.4.1, here with CUDA 12.1), in their own Python.
PY = "/content/rvc-env/bin/python"
if not os.path.exists(PY):
    sh("pip install -q uv")
    sh("uv venv -q --python 3.12 /content/rvc-env")
    sh("uv pip install -q --python %s torch==2.4.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu121" % PY)
    sh("uv pip install -q --python %s numpy==1.26.4 scipy==1.17.1 librosa==0.10.2.post1 numba==0.68.0 "
       "soundfile==0.14.0 av==15.1.0 ffmpeg-python==0.2.0 praat-parselmouth==0.4.7 transformers==4.49.0 "
       "huggingface_hub==0.36.2 tensorboard==2.21.0 matplotlib==3.11.2 scikit-learn==1.9.1 tqdm "
       "onnx==1.17.0 onnxruntime==1.20.1 speechbrain==1.0.3" % PY)
check = subprocess.run([PY, "-c", "import torch, speechbrain; print(torch.__version__, torch.cuda.is_available())"],
                       capture_output=True, text=True)
print("PyTorch", check.stdout.strip() or check.stderr[-500:])
if not check.stdout.strip().endswith("True"):
    raise SystemExit("PyTorch can't see the GPU - check step 1.")
print("OK - software installed")

In [ ]:
#@title 4. Trainer, base models and content encoder (2 - 5 minutes the first time)
RVC = "/content/rvc"
sh("%s %s/colab_queue.py setup --work %s --rvc %s --python %s" % (PY, TOOLS, WORK, RVC, PY), quiet=False)

In [ ]:
#@title 5. Training data (first time: ~25 minutes for VocalSet; afterwards instant)
sh("%s %s/colab_queue.py data --work %s --kit '%s' --python %s %s" % (
    PY, TOOLS, WORK, KIT, PY, "--vocalset" if INCLUDE_VOCALSET else ""), quiet=False)
sh("%s %s/colab_queue.py queue --work %s --rvc %s --python %s --precision %s --batch %d %s" % (
    PY, TOOLS, WORK, RVC, PY, PRECISION, BATCH_SIZE, "--vocalset" if INCLUDE_VOCALSET else ""), quiet=False)

In [ ]:
#@title 6. Train (runs until every voice is finished; keep this tab open)
#@markdown Shows the progress every 2 minutes. You can stop this cell any time; training keeps going in the
#@markdown background while Colab stays connected. Run all again after a disconnect to continue.
import re, time, subprocess
from IPython.display import clear_output
QUEUE = TOOLS + "/train_queue.py"
print(subprocess.run([PY, QUEUE, "resume", "--work", WORK], capture_output=True, text=True).stdout)
while True:
    time.sleep(120)
    st = subprocess.run([PY, QUEUE, "status", "--work", WORK], capture_output=True, text=True).stdout
    clear_output(wait=True)
    print(time.strftime("%H:%M:%S"), "\n")
    print(st.split("Resume (idempotent)")[0])
    if "queue complete" in st or "supervisor: not running" in st:
        log = open(WORK + "/queue_logs/supervisor.log").read()[-1500:]
        print("--- supervisor log ---\n" + log)
        break

In [ ]:
#@title 7. Finished voices: listen and download
import glob, json, os
from IPython.display import Audio, display, Markdown
state = json.load(open(WORK + "/queue_state.json")) if os.path.exists(WORK + "/queue_state.json") else {}
done = [n for n, s in state.items() if s.get("status") == "done"]
print("%d voices finished. The .vxpack files are in My Drive/VoxaVersa/queue/voices/<voice>/" % len(done))
for name in done:
    s = state[name]
    display(Markdown("**%s** - %s - powers: %s" % (name, s.get("final_summary", ""), ", ".join(s.get("factory_voices", [])) or "-")))
    for mp3 in sorted(glob.glob(os.path.join(WORK, "listen", name, "*_3_before_then_after.mp3"))):
        print(os.path.basename(mp3))
        display(Audio(mp3))
if os.path.exists(WORK + "/factory_assignment.md"):
    display(Markdown(open(WORK + "/factory_assignment.md").read()))

## Using the voices in VoxaVersa

Download the folder *My Drive > VoxaVersa > queue > voices* (right-click > Download), then follow
**docs/COLAB.md, part 3**: install the content encoder `contentvec-768.onnx` (in *My Drive > VoxaVersa >
queue*) once, and import each `.vxpack` in VoxaVersa's Voice Library. Each pack already says which built-in
genre voices it powers.